# SCOPE: train and evaluate on Colab

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

This notebook generates the synthetic visit notes, scores the rule baseline, fine-tunes two encoders
(`bert-base-uncased` and the clinical-notes model `emilyalsentzer/Bio_ClinicalBERT`), compares them on
three test sets, and publishes the better model to the Hugging Face Hub for the Streamlit app.
Each training run takes roughly 10–15 minutes on a T4.

In [ ]:
!git clone https://github.com/FabcanK6/scope.git
%cd scope
!pip install -q "transformers>=4.40" sentence-transformers faiss-cpu

## 1. Generate the data
12,000 training notes, plus validation, a seen-phrasing test set, an unseen-phrasing test set, and a bank of 400 past visits for search.

In [ ]:
!python -m scope.data.generate --out data

## 2. Rule baseline (regex + keywords + negation cues)

In [ ]:
!python -m scope.evaluate --backend rules --report reports/rules.json --quiet

## 3. Fine-tune bert-base-uncased

In [ ]:
!python -m scope.train --model bert-base-uncased --out models/scope-bert --epochs 3 --fp16

In [ ]:
!python -m scope.evaluate --model models/scope-bert --report reports/bert.json --quiet

## 4. Fine-tune Bio_ClinicalBERT (pre-trained on clinical notes)

In [ ]:
!python -m scope.train --model emilyalsentzer/Bio_ClinicalBERT --out models/scope-clinicalbert --epochs 3 --fp16

In [ ]:
!python -m scope.evaluate --model models/scope-clinicalbert --report reports/clinicalbert.json --quiet

## 5. Compare

In [ ]:
import json, pandas as pd
from scope.metrics import headline
rows = []
for name in ["rules", "bert", "clinicalbert"]:
    rep = json.load(open(f"reports/{name}.json"))
    for split, r in rep["sets"].items():
        rows.append({"parser": name, "test set": split, **{k: round(v, 3) for k, v in headline(r).items()}})
table = pd.DataFrame(rows).set_index(["test set", "parser"]).sort_index()
table

Look at a few mistakes on the hand-written notes (the hardest set):

In [ ]:
!python -m scope.evaluate --model models/scope-bert --data handwritten --errors 5 --quiet

## 6. Similar-visit search: embeddings vs TF-IDF
A hit counts as relevant when it shares an active issue with the query note.

In [ ]:
!python -m scope.search --eval

## 7. Publish the model for the app

The app downloads `FabcanK6/scope-bert` from the Hugging Face Hub. If Bio_ClinicalBERT scored better in step 5,
change `folder` below to `models/scope-clinicalbert` (the repo name stays `scope-bert`, so the app needs no change).

You need a Hugging Face **write** token: huggingface.co → Settings → Access Tokens → Create new token → *Write*.

In [ ]:
from huggingface_hub import login, HfApi
login()  # paste the write token when asked
folder = "models/scope-bert"
api = HfApi()
api.create_repo("FabcanK6/scope-bert", repo_type="model", exist_ok=True)
api.upload_folder(folder_path=folder, repo_id="FabcanK6/scope-bert", repo_type="model")
print("uploaded", folder)

## 8. Download the reports
Send these to update the README results table.

In [ ]:
!zip -qr scope-reports.zip reports models/scope-bert/training_history.json models/scope-clinicalbert/training_history.json
from google.colab import files
files.download("scope-reports.zip")